# MÔ HÌNH FT CÓ HƯỞNG LỢI TỪ VIỆC ĐỌC NHIỀU KHÔNG?

## Câu hỏi duy nhất
Mô hình **gốc** đi từ **0.6667** (3 đoạn, top-5) lên **0.7100** (50 đoạn, 2 góc nhìn) — **+4,33
điểm chỉ nhờ được đọc nhiều hơn**. Mô hình **FT** ở 3 đoạn đã đạt **0.7133**.
Nếu nó cũng hưởng lợi tương đương ⇒ **~0.757**, vượt xa GoGo 0.7210.
Nếu không ⇒ 0.7133 so với 0.702 hiện tại là quá sát, **không đáng nộp**.

## Thiết kế
Chấm **một lần** ở mức giàu nhất (top-10 văn bản × tối đa 10 đoạn) rồi **suy ra mọi cấu hình
nhỏ hơn từ cùng bộ điểm** — N ∈ {3,5,10} × k ∈ {1,3,5,10}, miễn phí, không chấm lại.

**Có nhánh ĐỐI CHỨNG**: chấm mô hình **gốc** trên ĐÚNG harness đó. Thiếu nó thì không phân biệt
được "mô hình FT giỏi hơn" với "harness giàu hơn thì ai cũng lên" — đúng lỗi đã làm hỏng
kết luận `enrich` và suýt hỏng kết luận `headview`.

## Ngưỡng đặt TRƯỚC — số quyết định là **FT tại N=10, k=10**
| kết quả | quyết định |
|---|---|
| **≥ 0,755** | FT hưởng lợi như bản gốc → chạy public, nộp |
| 0,72 – 0,755 | có lên nhưng chưa chắc vượt 0,7210 → cân nhắc, có thể nộp 1 bài thăm dò |
| **< 0,72** | FT KHÔNG hưởng lợi từ đọc nhiều → giữ 3 đoạn, và 0.7133 vs 0.702 quá sát, **không nộp** |

Bảng N×k in ra để xem **hình dạng đường cong**, KHÔNG phải để chọn cấu hình đẹp nhất.
Đường cong tăng đều theo k = cơ chế thật. Nhảy loạn = nhiễu.

## Cần Add Input
1. Dataset `Project_IR` (dev300, scores, selected-contexts, deep_chunk.py…)
2. **Output của notebook fine-tune** (hoặc dataset chứa thư mục `ft_listwise`)

GPU T4 · ~1,8h (2 mô hình × ~27.000 cặp).

In [ ]:
import os, sys, json, time, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

assert torch.cuda.is_available(), (
    "CHUA BAT GPU. Panel ben phai -> Settings -> Accelerator -> GPU T4 x2, roi Run All lai. "
    "(Kaggle nap image CPU khi Accelerator = None, torch khong co CUDA.)")
print(f"  GPU: {torch.cuda.get_device_name(0)}")

ROOT="/kaggle/input"
def find(name):
    for r,_,fs in os.walk(ROOT):
        if name in fs: return os.path.join(r,name)
    raise FileNotFoundError(f"KHONG THAY {name} duoi {ROOT}")
def find_model_dir():
    """Tim thu muc model bat ke Kaggle giu nguyen ft_listwise/ hay lam phang ra goc."""
    cands=[]
    for r,_,fs in os.walk(ROOT):
        if "config.json" in fs and any(
            f.endswith((".safetensors",".bin")) and f.startswith(("model","pytorch_model")) for f in fs):
            cands.append(r)
    if not cands:
        raise FileNotFoundError(f"KHONG THAY thu muc model nao duoi {ROOT} — da Add Input dataset chua?")
    for c in cands:                                   # uu tien ten dung
        if os.path.basename(c)=="ft_listwise": return c
    for c in cands:                                   # roi den duong dan co 'finetune'
        if "finetune" in c.lower(): return c
    if len(cands)==1: return cands[0]
    raise FileNotFoundError(f"KHONG XAC DINH duoc model. Ung vien: {cands}")

FT = find_model_dir()
print(f"  [model FT] {FT}")
print(f"  [chua]     {sorted(os.listdir(FT))[:6]}")
DEVF   = find("dev_300_locked.json")
SCORESF= find("scores_dev300_fusion_M20_K20.json")
UTIL   = os.path.dirname(find("deep_chunk.py"))
sys.path.insert(0, UTIL)
import deep_chunk as DC
DC.MERGE_CHARS = 1800
CTX=None
for _r,_,_fs in os.walk(ROOT):
    if any(f.startswith("context_") and f.endswith(".json") for f in _fs): CTX=_r; break
assert CTX, "KHONG THAY context_*.json"
assert len(DC.read_passage(CTX, os.listdir(CTX)[0][8:-5])) > 50, "CTX sai thu muc"
for n,v in [("FT",FT),("DEV",DEVF),("SCORES",SCORESF),("UTIL",UTIL),("CTX",CTX)]: print(f"  {n:<7} {v}")

BASE_MODEL = "AITeamVN/Vietnamese_Reranker"
MAXLEN, N_DOC, K_CHUNK = 1024, 10, 10
DOI_CHUNG = True    # False -> bo nhanh mo hinh goc, 105 phut con ~52 phut
OUT = "/kaggle/working/rich_harness_dev300.json"

dev=json.load(open(DEVF,encoding="utf-8")); S=json.load(open(SCORESF,encoding="utf-8"))
gold={q:{str(x) for x in v["answer"]} for q,v in dev.items()}; Q=list(gold)
mx=lambda v: max(v["ce"], v.get("ce_deep",-9e9))
order={q:[d for d,_ in sorted(S[q].items(), key=lambda kv:-mx(kv[1]))] for q in Q}
PROD=np.mean([order[q][0] in gold[q] for q in Q])
assert abs(PROD-0.7100)<1e-6, "SAI FILE DIEM"
for N in (5,10):
    print(f"  tran top-{N} = {np.mean([bool(gold[q]&set(order[q][:N])) for q in Q]):.4f}")
print(f"  san xuat (50 van ban, 2 goc nhin) = {PROD:.4f}")
try:
    tok = AutoTokenizer.from_pretrained(FT)
    print("  tokenizer: nap tu thu muc FT")
except Exception as e:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    print(f"  tokenizer: thu muc FT khong co -> dung cua {BASE_MODEL}")

In [ ]:
# ===== Cham CA HAI mo hinh tren cung harness. Luu tung doan -> suy ra moi cau hinh sau. =====
res = json.load(open(OUT,encoding="utf-8")) if os.path.isfile(OUT) else {}

def score_model(tag, path):
    if tag in res and len(res[tag])==len(Q):
        print(f"{tag}: da co, bo qua"); return
    m = AutoModelForSequenceClassification.from_pretrained(path).cuda().eval()
    out = res.get(tag, {}); t0=time.time(); npair=0
    with torch.no_grad():
        for i,q in enumerate(Q,1):
            if q in out: continue
            qt=dev[q]["question"]; per={}
            for d in order[q][:N_DOC]:
                ck=DC.pick_chunks(qt,CTX,d,k=K_CHUNK)
                if not ck: per[d]=[-9e9]; continue
                e=tok([qt]*len(ck), ck, truncation=True, max_length=MAXLEN,
                      padding=True, return_tensors="pt")
                with torch.autocast("cuda",dtype=torch.float16):
                    s=m(**{k:v.cuda() for k,v in e.items()}).logits.view(-1)
                per[d]=[float(x) for x in s]       # GIU tung doan theo dung thu tu pick_chunks
                npair+=len(ck)
            out[q]=per
            if i%50==0:
                res[tag]=out; json.dump(res, open(OUT,"w",encoding="utf-8"))
                el=time.time()-t0
                print(f"  {tag} {i}/{len(Q)} · {npair:,} cap · {el/60:.1f} phut · con ~{el/i*(len(Q)-i)/60:.0f} phut", flush=True)
    res[tag]=out; json.dump(res, open(OUT,"w",encoding="utf-8"))
    print(f"{tag}: XONG · {npair:,} cap · {(time.time()-t0)/60:.1f} phut", flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

if DOI_CHUNG: score_model("goc", BASE_MODEL)
score_model("ft",  FT)
print(f"\nDA LUU {OUT} — TAI VE TRUOC KHI DONG PHIEN")

In [ ]:
# ===== Suy ra bang N x k tu cung bo diem, khong cham lai =====
from math import comb
res=json.load(open(OUT,encoding="utf-8"))
TAGS=[t for t in ("goc","ft") if t in res]
assert "ft" in TAGS and all(len(res[t])==len(Q) for t in TAGS)

def pick(tag,N,k):
    o={}
    for q in Q:
        best,bs=None,-9e9
        for d in order[q][:N]:
            v=res[tag][q].get(d)
            if not v: continue
            s=max(v[:k])
            if s>bs: bs,best=s,d
        o[q]=best
    return o
acc=lambda o: np.mean([o[q] in gold[q] for q in Q])
def mc(a,b):
    A=np.array([a[q] in gold[q] for q in Q]); B=np.array([b[q] in gold[q] for q in Q])
    w=int((A&~B).sum()); l=int((B&~A).sum()); n=w+l
    p=(sum(comb(n,i) for i in range(max(w,l),n+1))/2**n*2) if n else 1.0
    return w,l,min(p,1.0)

KS=(1,3,5,10); NS=(3,5,10)
for tag in TAGS:
    print(f"\n=== {tag.upper()} ===   (hang = so van ban N · cot = so doan k)")
    print("      " + "".join(f"k={k:<7}" for k in KS))
    for N in NS:
        print(f"  N={N:<3}" + "".join(f"{acc(pick(tag,N,k)):<9.4f}" for k in KS))

f=pick("ft",N_DOC,K_CHUNK); F=acc(f)
PRODpick={q:order[q][0] for q in Q}
wp,lp,pp=mc(f,PRODpick)                       # so voi SAN XUAT = phep so quyet dinh
print("\n"+"="*68)
print(f"  san xuat hien tai (50 van ban, 2 goc nhin, mo hinh goc) = {PROD:.4f}")
if "goc" in TAGS:
    g=pick("goc",N_DOC,K_CHUNK); G=acc(g); w,l,p=mc(f,g)
    print(f"  GOC  tai N={N_DOC}, k={K_CHUNK} = {G:.4f}")
    print(f"  FT   tai N={N_DOC}, k={K_CHUNK} = {F:.4f}   ({F-G:+.4f} so voi goc cung harness · thang {w} thua {l} p={p:.4f})")
else:
    print(f"  FT   tai N={N_DOC}, k={K_CHUNK} = {F:.4f}   (khong chay doi chung)")
print(f"  >>> FT so voi SAN XUAT: {F-PROD:+.4f}   McNemar thang {wp} thua {lp}  p={pp:.4f}")
print(f"  tran top-{N_DOC} = {np.mean([bool(gold[q]&set(order[q][:N_DOC])) for q in Q]):.4f}")
print("="*68)
# QUY TAC (Khim chot 08/09): he TANG la thuc thi. Quota nop 10/ngay, GPU moi la thu khan hiem.
# Ky luat khong nam o "co nen thu khong" ma nam o "giu cai nao lam bai chot".
if F > PROD:
    print(f"TANG {F-PROD:+.4f} -> CHAY PUBLIC bang mo hinh FT roi NOP. De LB phan xu.")
    if   F-PROD >= 0.03: print("   bien do lon, kha nang cao la that.")
    elif pp < 0.05:      print("   bien do nho nhung McNemar co y nghia -> dang tin.")
    else:                print("   bien do nam trong nhieu (~1,5 diem). Nop de BIET, dung de TIN.")
    print("\n   SAU KHI NOP: LB > 0.6712 recall thi giu; <= thi NOP LAI A_K50_max_DANGNOP.zip ngay.")
    print("   Bai dung cuoi ngay phai la bai co diem LB cao nhat, khong phai bai moi nhat.")
else:
    print(f"KHONG TANG ({F-PROD:+.4f}) -> giu cau hinh hien tai 0.702, KHONG nop.")
print("\nDoc duong cong theo k: tang deu = co che that · nhay loan = nhieu.")